# Notebook 2/3 — Channel Pruning + Fine-Tuning (HARD 2.5h time budget)

**এই notebook একবারে ONE config train করে** — width `r` আর channel-selection method একটা config cell-এ সেট করা, দুইবার আলাদা Kaggle session-এ চালিয়ে (magnitude vs snr_aware) তুলনা করবে।

কারণ: Notebook 1-এ দেখা গেছে batch=8-এ ৫০০ epoch লাগে ৯০+ ঘণ্টা — একটাই notebook-এ সব ৪-৫টা model train করা **সম্ভব না**। তাই scope ছোট রাখা, প্রতিটা run আলাদা ২-৩ ঘণ্টার মধ্যে শেষ, guaranteed।

### সময়ের বাজেট কীভাবে ভাগ করা
```
মোট বাজেট        : 2.5h (হার্ড cap, TimeBudgetCallback দিয়ে enforce করা)
  batch probe    : ~2 মিনিট
  channel select : ~5-10 মিনিট (magnitude সেকেন্ডে, snr_aware কয়েক মিনিট)
  fine-tuning    : বাকি সময় থেকে eval-reserve বাদ দিয়ে যতটা পাওয়া যায়
  evaluation     : ~20 মিনিট reserve (subsample করা eval bank দিয়ে)
```

### দুইবার চালানোর নিয়ম
```
Run 1: SELECTION_METHOD = 'magnitude'   (Cell 6-এ সেট করো)
Run 2: SELECTION_METHOD = 'snr_aware'   (Cell 6-এ বদলে আবার Run All)
```
দুটো run-এর result আলাদা ফাইলে save হবে, Notebook 3-এ বা পরে তুলনা করা যাবে।

### Kaggle-এ attach করতে হবে
- `dldoa-source-code` (DL_DOA folder + pretrained weights)
- `dldoa-frozen-banks` (eval_bank.npz + calibration_bank.npz)
- **নতুন:** repo-র root-এ থাকা `dldoa_dataset_generation.py` — এটাও `dldoa-source-code` dataset-এ থাকলে ভালো, না থাকলে আলাদা attach করো

In [ ]:
# Cell 1 — Setup + wall-clock start
T_START = None
import importlib, subprocess, sys, time
T_START = time.time()   # everything below is timed against this

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model

tf.keras.backend.clear_session()
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
for g in gpus: tf.config.experimental.set_memory_growth(g, True)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

def elapsed_hours():
    return (time.time() - T_START) / 3600

print(f'Wall clock started. t=0.00h')

In [ ]:
# Cell 2 — Locate repo source, weights, frozen banks, and the training-data generator script
def find_path(name_pattern):
    from pathlib import Path
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for p in Path(root).rglob(name_pattern):
            if p.is_file(): return str(p)
    return None

SRC_MODEL_FILE = find_path('tvt_models.py')
assert SRC_MODEL_FILE is not None, 'DL_DOA source not found -- attach dldoa-source-code'
DL_DOA_DIR = os.path.dirname(os.path.dirname(SRC_MODEL_FILE))
print(f'DL_DOA dir: {DL_DOA_DIR}')

WEIGHTS_PATH = find_path('inf_model_007_256_resnet.h5')
assert WEIGHTS_PATH is not None, 'pretrained weights not found'
print(f'Weights: {WEIGHTS_PATH}')

EVAL_BANK_PATH = find_path('eval_bank.npz')
CALIB_BANK_PATH = find_path('calibration_bank.npz')
assert EVAL_BANK_PATH is not None and CALIB_BANK_PATH is not None, 'frozen banks not found -- attach dldoa-frozen-banks'
print(f'Eval bank: {EVAL_BANK_PATH}')
print(f'Calibration bank: {CALIB_BANK_PATH}')

TRAIN_GEN_FILE = find_path('dldoa_dataset_generation.py')
assert TRAIN_GEN_FILE is not None, 'dldoa_dataset_generation.py not found -- needed for the training generator'
print(f'Training generator script: {TRAIN_GEN_FILE}')

In [ ]:
# Cell 3 — Import ORIGINAL evaluator + Resnet (metric-critical, never reimplemented)
# and the paper-faithful training generator (training-data variety isn't drift-
# sensitive the way the evaluation metric is, so this can be the standalone script).
sys.path.insert(0, DL_DOA_DIR)
sys.path.insert(0, os.path.dirname(TRAIN_GEN_FILE))

from src.tvt_models import Resnet
from src.TVT_Blob_Inference import (
    get_blob_detector, get_blob_peaks, peaks_to_angles,
    prepare_for_metric, get_ang_difference, filter_angles,
)
from dldoa_dataset_generation import training_data_generator

print('✅ Imported Resnet, original evaluator functions, and training_data_generator')

In [ ]:
# Cell 4 — Load frozen banks
eval_bank = np.load(EVAL_BANK_PATH)
EVAL_DATA, EVAL_FEAT, EVAL_META = eval_bank['data'], eval_bank['feat'], eval_bank['meta']
SIGMA = float(eval_bank['sigma']); M = int(eval_bank['M'])

calib_bank = np.load(CALIB_BANK_PATH)
CALIB_DATA = calib_bank['data']
CALIB_GT = calib_bank['gt'].astype(np.float32)   # was float16 on disk
CALIB_META = calib_bank['meta']

print(f'Eval bank: {EVAL_DATA.shape[0]} samples | Calibration bank: {CALIB_DATA.shape[0]} samples')

In [ ]:
# Cell 5 — Load teacher (frozen, used for distillation + comparison)
teacher = Resnet(input_shape=(64, 64, 2))
teacher.load_weights(WEIGHTS_PATH)
teacher.trainable = False
TEACHER_PARAMS = teacher.count_params()
print(f'✅ Teacher loaded: {TEACHER_PARAMS:,} params (frozen)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — CONFIG -- change these two lines between runs
# ══════════════════════════════════════════════════════════════
WIDTH_R          = 8              # internal bottleneck width (paper original = 12)
SELECTION_METHOD = 'magnitude'    # 'magnitude'  or  'snr_aware'
LAMBDA_DISTILL   = 0.5            # 0.0 = GT-only fine-tune, 0.5 = + teacher distillation

TOTAL_TIME_BUDGET_HOURS = 2.5     # HARD cap for this whole notebook run
EVAL_RESERVE_MINUTES    = 20      # time reserved for Cell 12's evaluation, not spent training
N_PER_SNR_EVAL          = 150     # subsampled eval size (of 1000 available) -- lower if time is tight
MAX_EPOCHS_CAP          = 200     # safety ceiling, time budget will stop training long before this

RUN_TAG = f'r{WIDTH_R}_{SELECTION_METHOD}_lambda{LAMBDA_DISTILL}'
print(f'Run config: {RUN_TAG}')
print(f'Time budget: {TOTAL_TIME_BUDGET_HOURS}h total, {EVAL_RESERVE_MINUTES}min reserved for eval')

In [ ]:
# Cell 7 — Pruned-ResNet builder + weight-copy function (same as Notebook 1, re-validated there)
def res_conv_pruned(x, r, out_filters=12):
    skip = x
    x = Conv2D(r, 5, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(out_filters, 5, padding='same')(x); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x

def build_pruned_resnet(r, n_blocks=64, input_shape=(64, 64, 2), name=None):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(n_blocks):
        x = res_conv_pruned(x, r)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=name or f'PrunedResNet-r{r}')

def get_weighted_layers(model):
    return [l for l in model.layers if l.get_weights()]

def copy_weights_to_pruned(teacher_model, student_model, filter_indices_per_block, n_blocks=64):
    """Conv2D kernel shape = (kh, kw, in_ch, out_ch). Conv1 slices axis=3 (output),
    Conv2 slices axis=2 (input). BN1 slices all 4 arrays; BN2 and outer transpose
    layers copy unchanged."""
    tw = get_weighted_layers(teacher_model)
    sw = get_weighted_layers(student_model)
    assert len(tw) == len(sw) == (1 + 4 * n_blocks + 1), f'{len(tw)} vs {len(sw)} weighted layers'

    sw[0].set_weights(tw[0].get_weights())
    for i in range(n_blocks):
        J = np.asarray(filter_indices_per_block[i])
        base = 1 + 4 * i
        t_conv1, t_bn1, t_conv2, t_bn2 = tw[base:base+4]
        s_conv1, s_bn1, s_conv2, s_bn2 = sw[base:base+4]

        k, b = t_conv1.get_weights()
        s_conv1.set_weights([k[:, :, :, J], b[J]])

        gamma, beta, mean, var = t_bn1.get_weights()
        s_bn1.set_weights([gamma[J], beta[J], mean[J], var[J]])

        k2, b2 = t_conv2.get_weights()
        s_conv2.set_weights([k2[:, :, J, :], b2])

        s_bn2.set_weights(t_bn2.get_weights())

    sw[-1].set_weights(tw[-1].get_weights())

print('✅ Pruned-ResNet builder + weight-copy function ready')

In [ ]:
# Cell 8 — Batch-size probe: find the largest batch that doesn't OOM at WIDTH_R
# (Notebook 1 was stuck at batch=8, which made 500-epoch training take 90+ hours.
#  If the crash-induced fragmentation is really gone after a clean session, a
#  bigger batch here directly buys back most of that lost time.)

def try_batch(r, batch):
    try:
        m = build_pruned_resnet(r, name='probe_batch_test')
        opt = tf.keras.optimizers.Adam(1e-4)
        x = tf.random.normal((batch, 64, 64, 2))
        y = tf.random.normal((batch, 256, 256, 1))
        with tf.GradientTape() as tape:
            pred = m(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, m.trainable_variables)
        opt.apply_gradients(zip(grads, m.trainable_variables))
        del m, opt
        tf.keras.backend.clear_session()
        return True
    except tf.errors.ResourceExhaustedError:
        tf.keras.backend.clear_session()
        return False

BATCH = 8   # safe fallback
for candidate in [32, 24, 16, 8]:
    print(f'Trying batch={candidate}...')
    if try_batch(WIDTH_R, candidate):
        BATCH = candidate
        print(f'✅ batch={candidate} works -- using this')
        break
    else:
        print(f'  OOM at batch={candidate}, trying smaller')

STEPS_PER_EPOCH = int(np.ceil(10000 / BATCH))
print(f'Final: BATCH={BATCH}, STEPS_PER_EPOCH={STEPS_PER_EPOCH}')
print(f'Elapsed so far: {elapsed_hours():.2f}h')

In [ ]:
# Cell 9 — Channel selection: magnitude-based (instant) and SNR-stratified Taylor-based

def select_channels_magnitude(teacher_model, r, n_blocks=64, in_channels=12):
    tw = get_weighted_layers(teacher_model)
    indices = []
    for i in range(n_blocks):
        base = 1 + 4 * i
        k1, _ = tw[base].get_weights()      # Conv1 kernel (5,5,12,12)
        k2, _ = tw[base + 2].get_weights()  # Conv2 kernel (5,5,12,12)
        score = np.array([
            np.linalg.norm(k1[:, :, :, c]) * np.linalg.norm(k2[:, :, c, :])
            for c in range(in_channels)
        ])
        indices.append(np.sort(np.argsort(-score)[:r]))
    return indices

def build_probe_resnet(n_blocks=64, input_shape=(64, 64, 2)):
    """Same topology as the r=12 teacher, but also outputs each block's
    post-Conv1-BN-ReLU activation, so Taylor importance can be computed."""
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    intermediates = []
    for _ in range(n_blocks):
        skip = x
        x = Conv2D(12, 5, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
        intermediates.append(x)
        x = Conv2D(12, 5, padding='same')(x); x = BatchNormalization()(x)
        x = Add()([x, skip]); x = Activation('relu')(x)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, [x] + intermediates, name='ProbeResNet-r12')

def compute_taylor_scores(probe_model, data, gt, n_blocks=64, batch_size=8):
    sums = [np.zeros(12) for _ in range(n_blocks)]
    N = data.shape[0]
    if N == 0: return sums
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        xb = tf.constant(data[start:end]); yb = tf.constant(gt[start:end])
        with tf.GradientTape() as tape:
            outs = probe_model(xb, training=False)
            pred, inter = outs[0], outs[1:]
            loss = tf.reduce_mean(tf.square(pred - yb))
        grads = tape.gradient(loss, inter)
        for i in range(n_blocks):
            imp = tf.reduce_mean(tf.abs(inter[i] * grads[i]), axis=[0, 1, 2]).numpy()
            sums[i] += imp * (end - start)
    return [s / N for s in sums]

def select_channels_snr_aware(teacher_model, calib_data, calib_gt, calib_meta, r, n_blocks=64):
    probe = build_probe_resnet(n_blocks)
    identity_indices = [list(range(12))] * n_blocks
    copy_weights_to_pruned(teacher_model, probe, identity_indices, n_blocks)

    low_mask = calib_meta[:, 1] <= 0     # SNR <= 0 dB -- hard band
    high_mask = ~low_mask
    print(f'  Taylor scoring: {low_mask.sum()} low-SNR + {high_mask.sum()} high-SNR calibration samples')
    scores_low  = compute_taylor_scores(probe, calib_data[low_mask],  calib_gt[low_mask],  n_blocks)
    scores_high = compute_taylor_scores(probe, calib_data[high_mask], calib_gt[high_mask], n_blocks)

    indices = []
    for i in range(n_blocks):
        sl = scores_low[i] / (scores_low[i].max() + 1e-12)
        sh = scores_high[i] / (scores_high[i].max() + 1e-12)
        combined = np.maximum(sl, sh)   # protects a channel that only matters in ONE band
        indices.append(np.sort(np.argsort(-combined)[:r]))
    del probe; tf.keras.backend.clear_session()
    return indices

print(f'Selecting channels via: {SELECTION_METHOD}')
if SELECTION_METHOD == 'magnitude':
    filter_indices = select_channels_magnitude(teacher, WIDTH_R)
elif SELECTION_METHOD == 'snr_aware':
    filter_indices = select_channels_snr_aware(teacher, CALIB_DATA, CALIB_GT, CALIB_META, WIDTH_R)
else:
    raise ValueError(f'Unknown SELECTION_METHOD: {SELECTION_METHOD}')

print(f'✅ Channel indices selected for {len(filter_indices)} blocks (r={WIDTH_R} of 12 each)')
print(f'Elapsed so far: {elapsed_hours():.2f}h')

In [ ]:
# Cell 10 — Build student, copy selected weights, report compression
student = build_pruned_resnet(WIDTH_R, name=f'Student-{RUN_TAG}')
copy_weights_to_pruned(teacher, student, filter_indices)

STUDENT_PARAMS = student.count_params()
print(f'Student params: {STUDENT_PARAMS:,}')
print(f'Teacher params: {TEACHER_PARAMS:,}')
print(f'Reduction: {(1 - STUDENT_PARAMS/TEACHER_PARAMS)*100:.1f}%')

In [ ]:
# Cell 11 — Fine-tune with GT + optional distillation loss, hard time-budget stop
def make_train_ds(batch):
    def fn():
        for d, g in training_data_generator(sigma=SIGMA, M=M):
            yield d, g
    ds = tf.data.Dataset.from_generator(
        fn,
        output_signature=(
            tf.TensorSpec(shape=(64, 64, 2), dtype=tf.float32),
            tf.TensorSpec(shape=(M, M, 1), dtype=tf.float32),
        ),
    )
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

opt = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        pred = student(x, training=True)
        loss_gt = tf.reduce_mean(tf.square(pred - y))
        if LAMBDA_DISTILL > 0:
            teacher_pred = teacher(x, training=False)
            loss_distill = tf.reduce_mean(tf.square(pred - teacher_pred))
        else:
            loss_distill = 0.0
        loss = loss_gt + LAMBDA_DISTILL * loss_distill
    grads = tape.gradient(loss, student.trainable_variables)
    opt.apply_gradients(zip(grads, student.trainable_variables))
    return loss

train_budget_hours = max(0.05, TOTAL_TIME_BUDGET_HOURS - elapsed_hours() - EVAL_RESERVE_MINUTES/60)
print(f'Training time budget: {train_budget_hours:.2f}h  (of {TOTAL_TIME_BUDGET_HOURS}h total, '
      f'{elapsed_hours():.2f}h already elapsed, {EVAL_RESERVE_MINUTES}min reserved for eval)')

ds_iter = iter(make_train_ds(BATCH))
train_start = time.time()
history_loss = []
epoch = 0
while epoch < MAX_EPOCHS_CAP:
    if (time.time() - train_start) / 3600 >= train_budget_hours:
        print(f'⏱️  Training time budget reached at epoch {epoch}')
        break
    epoch_losses = []
    for step in range(STEPS_PER_EPOCH):
        x, y = next(ds_iter)
        epoch_losses.append(float(train_step(x, y)))
        if (time.time() - train_start) / 3600 >= train_budget_hours:
            break
    history_loss.append(float(np.mean(epoch_losses)))
    epoch += 1
    print(f'Epoch {epoch}: loss={history_loss[-1]:.5f}  '
          f'elapsed={(time.time()-train_start)/60:.1f}min')

EPOCHS_COMPLETED = epoch
print(f'\n✅ Fine-tuning done: {EPOCHS_COMPLETED} epochs in {(time.time()-train_start)/60:.1f} min')
student.save_weights(os.path.join(OUT_DIR, f'student_{RUN_TAG}.weights.h5'))

plt.figure(figsize=(8, 3.5))
plt.plot(history_loss, color='crimson')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'Fine-tuning loss ({RUN_TAG})')
plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'train_loss_{RUN_TAG}.png'), dpi=130); plt.show()
print(f'Elapsed so far: {elapsed_hours():.2f}h  (budget: {TOTAL_TIME_BUDGET_HOURS}h)')

In [ ]:
# Cell 12 — Evaluate student vs teacher on a SUBSAMPLED slice of the frozen eval bank
# (subsampled, not the full 8000, to guarantee this fits the reserved time -- the
# eval bank is ordered in 1000-sample blocks per SNR, so "first N per block" is a
# clean, reproducible slice used identically for every run/config)

def subsample_per_snr(data, feat, meta, n_per_snr, block_size=1000, n_blocks=8):
    idx = np.concatenate([np.arange(i*block_size, i*block_size+n_per_snr) for i in range(n_blocks)])
    return data[idx], feat[idx], meta[idx]

SUB_DATA, SUB_FEAT, SUB_META = subsample_per_snr(EVAL_DATA, EVAL_FEAT, EVAL_META, N_PER_SNR_EVAL)
print(f'Evaluating on {SUB_DATA.shape[0]} samples ({N_PER_SNR_EVAL}/SNR)')

def evaluate_on_bank(model, data_arr, feat_arr, meta_arr, batch_size=8, max_deg_error=1.0):
    detector = get_blob_detector()
    results_by_snr = {}
    N = data_arr.shape[0]
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        preds = model(data_arr[start:end], training=False)
        for j in range(end - start):
            idx = start + j
            L = int(meta_arr[idx, 0]); snr = int(meta_arr[idx, 1])
            peaks, amps = get_blob_peaks(preds[j], detector)
            order = np.argsort(-amps); peaks = peaks[order[:L]]
            angles_est = peaks_to_angles(peaks, sigma=SIGMA, grid_size=M)
            gt_angles, pred_angles = prepare_for_metric(angles_est, feat_arr[idx])
            results_by_snr.setdefault(snr, []).append((gt_angles, pred_angles))

    final_pd, final_rmse = {}, {}
    for snr, examples in results_by_snr.items():
        good_all, bad_all = [], []
        for gt, pred in examples:
            if np.isnan(pred).any(): continue
            diffs = get_ang_difference(gt, pred)
            good, bad = filter_angles(diffs, max_deg_error=max_deg_error)
            good_all.append(good); bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        final_pd[snr] = len(good_all)/total if total > 0 else np.nan
        final_rmse[snr] = np.sqrt(np.mean(good_all**2)) if len(good_all) > 0 else np.nan
    return final_pd, final_rmse

student_pd, student_rmse = evaluate_on_bank(student, SUB_DATA, SUB_FEAT, SUB_META)
teacher_pd, teacher_rmse = evaluate_on_bank(teacher, SUB_DATA, SUB_FEAT, SUB_META)
print(f'\n✅ Evaluation complete. Elapsed: {elapsed_hours():.2f}h')

In [ ]:
# Cell 13 — Compare student vs teacher + save results
snrs_sorted = sorted(teacher_pd.keys())
print('='*80)
print(f'Run: {RUN_TAG}   ({SUB_DATA.shape[0]} eval samples)')
print('='*80)
print(f'{"SNR":>5} | {"Student Pd":>11} {"Teacher Pd":>11} {"ΔPd":>8} | {"Stud RMSE":>10} {"Teach RMSE":>11}')
print('-'*80)
for s in snrs_sorted:
    dpd = student_pd[s] - teacher_pd[s]
    print(f'{s:>5} | {student_pd[s]:>11.4f} {teacher_pd[s]:>11.4f} {dpd:>+8.4f} | '
          f'{student_rmse[s]:>10.4f} {teacher_rmse[s]:>11.4f}')
print('='*80)
mean_dpd = np.mean([student_pd[s]-teacher_pd[s] for s in snrs_sorted])
mean_dpd_low = np.mean([student_pd[s]-teacher_pd[s] for s in snrs_sorted if s <= 0])
print(f'Mean ΔPd (all SNR): {mean_dpd:+.4f}')
print(f'Mean ΔPd (low SNR <=0dB): {mean_dpd_low:+.4f}  <- this is what SNR-aware selection is meant to protect')

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].plot(snrs_sorted, [student_pd[s] for s in snrs_sorted], 'o-', color='crimson', label=f'Student ({RUN_TAG})')
axs[0].plot(snrs_sorted, [teacher_pd[s] for s in snrs_sorted], 's--', color='steelblue', label='Teacher')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('Pd'); axs[0].set_title('Pd'); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].plot(snrs_sorted, [student_rmse[s] for s in snrs_sorted], 'o-', color='crimson', label=f'Student ({RUN_TAG})')
axs[1].plot(snrs_sorted, [teacher_rmse[s] for s in snrs_sorted], 's--', color='steelblue', label='Teacher')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('RMSE (deg)'); axs[1].set_title('RMSE'); axs[1].legend(); axs[1].grid(alpha=.3)
plt.suptitle(f'{RUN_TAG}  |  {STUDENT_PARAMS:,} vs {TEACHER_PARAMS:,} params '
             f'({(1-STUDENT_PARAMS/TEACHER_PARAMS)*100:.0f}% smaller)', y=1.03)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'compare_{RUN_TAG}.png'), dpi=140, bbox_inches='tight')
plt.show()

results = {
    'run_tag': RUN_TAG, 'width_r': WIDTH_R, 'selection_method': SELECTION_METHOD,
    'lambda_distill': LAMBDA_DISTILL, 'batch': BATCH, 'epochs_completed': EPOCHS_COMPLETED,
    'n_per_snr_eval': N_PER_SNR_EVAL,
    'student_params': STUDENT_PARAMS, 'teacher_params': TEACHER_PARAMS,
    'student_pd': {str(k): float(v) for k, v in student_pd.items()},
    'student_rmse': {str(k): float(v) for k, v in student_rmse.items()},
    'teacher_pd': {str(k): float(v) for k, v in teacher_pd.items()},
    'teacher_rmse': {str(k): float(v) for k, v in teacher_rmse.items()},
    'mean_dpd_all': float(mean_dpd), 'mean_dpd_low_snr': float(mean_dpd_low),
    'total_elapsed_hours': float(elapsed_hours()),
}
result_path = os.path.join(OUT_DIR, f'results_{RUN_TAG}.json')
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {result_path}')

In [ ]:
# Cell 14 — Final summary
print('='*70)
print('NOTEBOOK 2 RUN SUMMARY')
print('='*70)
print(f'Config:            {RUN_TAG}')
print(f'Total elapsed:     {elapsed_hours():.2f}h  (budget: {TOTAL_TIME_BUDGET_HOURS}h)')
print(f'{"✅ within budget" if elapsed_hours() <= TOTAL_TIME_BUDGET_HOURS + 0.1 else "⚠️ exceeded budget"}')
print(f'Epochs completed:  {EPOCHS_COMPLETED}')
print(f'Params:            {STUDENT_PARAMS:,} ({(1-STUDENT_PARAMS/TEACHER_PARAMS)*100:.1f}% smaller than teacher)')
print(f'Mean ΔPd (all):    {mean_dpd:+.4f}')
print(f'Mean ΔPd (low SNR):{mean_dpd_low:+.4f}')
print()
print('পরের ধাপ:')
if SELECTION_METHOD == 'magnitude':
    print('  Cell 6-এ SELECTION_METHOD = \'snr_aware\' করে আবার Run All দাও (নতুন Kaggle session-এ)')
    print('  তারপর results_r{}_magnitude...json vs results_r{}_snr_aware...json তুলনা করা যাবে')
else:
    print('  magnitude run-এর result-এর সাথে এই snr_aware result তুলনা করো')
    print('  (দুটো results_*.json ফাইল Notebook 3-এ বা manually পাশাপাশি রাখো)')